# CPRD GOLD: Source Overview and Exploratory Data Analysis

**CPRD GOLD** (Clinical Practice Research Datalink) is a UK primary-care database derived from anonymised GP records. This notebook walks through the canonical Python workflow for CPRD data using `ehrdata.io.source.adapters.cprd` — the direct Python translation of the original R EDA pipeline.

**Topics covered**
1. Loading raw CPRD zip archives into DataFrames
2. Patient and event-date EDA across file types
3. Read code hierarchy analysis (diagnosis)
4. Drug-substance therapy summary
5. Qualified-patient filtering rules

In [1]:
import io
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from ehrdata.io.source.adapters import cprd
from ehrdata.io.source.vocab import readcode, prodcode

## 1. Loading Raw CPRD Data

CPRD distributes data as zip archives of tab-delimited `.txt` files. Each file type (Clinical, Referral, Test, Additional, Therapy, Patient) may span multiple `.zip` files.

The `extract.read_zipped_tsvs` helper reads them directly without unpacking. Below we build synthetic DataFrames that mirror the real CPRD schema so you can follow along without access to raw data.

In [2]:
# --- Synthetic CPRD DataFrames (mirrors real schema) ---

clinical = pd.DataFrame({
    "patid":     ["P001", "P001", "P002", "P003", "P003"],
    "eventdate": ["01/06/2010", "15/08/2012", "03/03/2008", "22/11/2015", "22/11/2015"],
    "medcode":   [100, 200, 100, 300, 300],
    "enttype":   [4, 4, 4, 5, 5],
    "adid":      ["A1", "A2", "A3", "A4", "A4"],
})

referral = pd.DataFrame({
    "patid":     ["P002", "P003"],
    "eventdate": ["14/07/2009", "01/01/2016"],
    "medcode":   [200, 400],
    "enttype":   [3, 3],
    "adid":      ["", ""],
})

test_data = pd.DataFrame({
    "patid":     ["P001", "P002", "P003"],
    "eventdate": ["10/05/2011", "20/09/2013", "01/04/2017"],
    "medcode":   [500, 100, 200],
    "enttype":   [4, 5, 4],
    "adid":      ["A5", "A6", "A7"],
    "data1": [1.0, 2.0, 3.0], "data2": [7.1, 6.4, 8.9],
    "data3": ["%", "%", "%"],  "data4": ["", "", ""],
    "data5": [None, None, None], "data6": [None, None, None], "data7": [None, None, None],
})

additional = pd.DataFrame({
    "patid":   ["P001", "P002", "P003"],
    "enttype": [4, 4, 5],
    "adid":    ["A1", "A3", "A4"],
    "data1": [1.0, 1.0, 2.0], "data2": [7.1, 6.4, 8.9],
    "data3": ["%", "%", "%"],  "data4": ["", "", ""],
    "data5": [None, None, None], "data6": [None, None, None], "data7": [None, None, None],
})

therapy_data = pd.DataFrame({
    "patid":     ["P001", "P001", "P002", "P003"],
    "eventdate": ["10/01/2010", "20/03/2011", "05/06/2009", "12/12/2014"],
    "prodcode":  ["PROD1", "PROD1", "PROD2", "PROD3"],
    "bnfcode":   ["6.1.2", "6.1.2", "6.1.1", "2.12"],
    "qty":       [28, 28, 56, 28],
    "issueseq":  [1, 2, 1, 1],
})

patient = pd.DataFrame({
    "patid":     ["P001", "P002", "P003"],
    "yob":       [155, 162, 178],   # encoded as year_of_birth - 1800
    "sex":       ["Female", "Male", "Female"],
    "pracid":    ["PR1", "PR1", "PR2"],
    "gender":    [2, 1, 2],
    "frd":       ["01/01/2000", "01/06/1998", "15/03/2010"],
    "tod":       [None, None, None],
    "deathdate": [None, None, None],
})

print(f"Clinical rows  : {len(clinical)}")
print(f"Referral rows  : {len(referral)}")
print(f"Test rows      : {len(test_data)}")
print(f"Additional rows: {len(additional)}")
print(f"Therapy rows   : {len(therapy_data)}")
print(f"Patient rows   : {len(patient)}")

Clinical rows  : 5
Referral rows  : 2
Test rows      : 3
Additional rows: 3
Therapy rows   : 4
Patient rows   : 3


## 2. Vocabulary Maps

CPRD uses two internal code systems that must be translated:

| Code | Meaning | Vocab file |
|---|---|---|
| `medcode` | Integer → Read code (diagnosis) | `medical.txt` |
| `prodcode` | String → drug substance (therapy) | `product.txt` / `product.csv` |

Use `readcode.load_medical_map` and `prodcode.load_product_map` to load them.

In [3]:
# Synthetic vocabulary maps (in practice, load from medical.txt / product.csv)
medical_map_df = pd.DataFrame({
    "medcode":  [100, 200, 300, 400, 500],
    "readcode": ["A10..00", "C10..00", "E12..00", "F45..00", "44J3.00"],
    "desc":     ["Cholera", "Diabetes mellitus", "Obesity", "Anxiety disorder", "HbA1c"],
})

product_map_df = pd.DataFrame({
    "prodcode":            ["PROD1", "PROD2", "PROD3"],
    "drugsubstance":       ["metformin", "insulin glargine", "atorvastatin"],
    "drugsubstance.updated": ["metformin", "insulin glargine", "atorvastatin"],
})

# Use the loader functions on in-memory CSV to demonstrate the API
import tempfile, os

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    medical_map_df.to_csv(f, sep="\t", index=False)
    med_path = f.name

with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False) as f:
    product_map_df.to_csv(f, index=False)
    prod_path = f.name

medical_map = readcode.load_medical_map(med_path)
product_map  = prodcode.load_product_map(prod_path)

os.unlink(med_path)
os.unlink(prod_path)

print("Medical map (medcode → readcode):")
print(medical_map.head(3).to_string(index=False))
print()
print("Product map (prodcode → drugsubstance):")
print(product_map.head(3).to_string(index=False))

Medical map (medcode → readcode):
medcode readcode
    100  A10..00
    200  C10..00
    300  E12..00

Product map (prodcode → drugsubstance):
prodcode    drugsubstance
   PROD1        metformin
   PROD2 insulin glargine
   PROD3     atorvastatin


## 3. Building Canonical Tables

The adapter exposes four builder functions that normalise each domain into a standard schema. Pass optional vocab maps to translate internal codes.

In [4]:
# Diagnosis: Clinical + Referral + Test → one row per code per patient per date
diag = cprd.build_diagnosis(clinical, referral, test_data, medical_map=medical_map)
print(f"Diagnosis rows: {len(diag)}")
print(diag.dtypes)
diag.head()

Diagnosis rows: 9
patient_id            object
dxver                 object
eventdate     datetime64[ns]
dx                    object
dtype: object


,patient_id,dxver,eventdate,dx
0,P001,None,2010-06-01,A10..00
1,P001,None,2011-05-10,44J3.00
2,P001,None,2012-08-15,C10..00
3,P002,None,2008-03-03,A10..00
4,P002,None,2009-07-14,C10..00


In [5]:
# Therapy: prodcode translated to drug substance
therapy = cprd.build_therapy(therapy_data, product_map=product_map)
print(f"Therapy rows: {len(therapy)}")
therapy[["patient_id", "fill_date", "ingredient", "rxcui", "ndc11"]]

Therapy rows: 4


,patient_id,fill_date,ingredient,rxcui,ndc11
0,P001,2010-01-10,metformin,None,None
1,P001,2011-03-20,metformin,None,None
2,P002,2009-06-05,insulin glargine,None,None
3,P003,2014-12-12,atorvastatin,None,None


In [6]:
# Lab test: Clinical + Additional inner-joined on (patid, adid, enttype) → eventdate recovered
# Then unioned with Test files that already contain eventdate
labs = cprd.build_labtest(clinical, additional, test_data)
print(f"Lab test rows: {len(labs)}")
labs[["patient_id", "eventdate", "value", "unit", "loinc"]]

Lab test rows: 6


,patient_id,eventdate,value,unit,loinc
0,P001,2010-06-01,7.1,%,None
1,P001,2011-05-10,7.1,%,None
2,P002,2008-03-03,6.4,%,None
3,P002,2013-09-20,6.4,%,None
4,P003,2015-11-22,8.9,%,None
5,P003,2017-04-01,8.9,%,None


In [7]:
# Patient info: three canonical columns only
patinfo = cprd.build_patinfo(patient)
patinfo

,patient_id,dobyr,sex
0,P001,155,Female
1,P002,162,Male
2,P003,178,Female


## 4. Patient and Event-Date EDA

The original R pipeline computed unique `(patid, eventdate)` pairs across all file types to characterise the temporal footprint of the extract. The Python equivalent uses `pd.concat` + `.drop_duplicates()`.

In [8]:
_DATE_FMT = "%d/%m/%Y"

def _patid_dates(df, date_col="eventdate"):
    return (
        df[["patid", date_col]]
        .rename(columns={date_col: "eventdate"})
        .assign(eventdate=lambda d: pd.to_datetime(d["eventdate"], format=_DATE_FMT, errors="coerce"))
        .drop_duplicates()
    )

all_events = pd.concat([
    _patid_dates(clinical),
    _patid_dates(referral),
    _patid_dates(test_data),
    _patid_dates(therapy_data),
], ignore_index=True).drop_duplicates()

print(f"Total unique (patid, eventdate) pairs : {len(all_events):,}")
print(f"Unique patients                        : {all_events['patid'].nunique():,}")
print(f"Date range                             : {all_events['eventdate'].min().date()} → {all_events['eventdate'].max().date()}")
all_events.groupby(all_events["eventdate"].dt.year).size().rename("events").to_frame().head(10)

Total unique (patid, eventdate) pairs : 13
Unique patients                        : 3
Date range                             : 2008-03-03 → 2017-04-01


,events
eventdate,
2008,1
2009,2
2010,2
2011,2
2012,1
2013,1
2014,1
2015,1
2016,1


## 5. Read Code Hierarchy Analysis

CPRD Read codes follow a five-character hierarchy. The first character indicates the major chapter (A = Infectious diseases, C = Endocrine, E = Mental health, etc.). Truncating to the first two characters (e.g. `C10...`) gives the sub-chapter level.

The R pipeline matched patients to pre-defined code lists at this truncated level.

In [9]:
# Merge diagnosis with the medical map to get read codes
diag_coded = diag.copy()

# Filter: exclude codes beginning with 'R' (symptoms) or 'ZZ' (unmapped)
diag_coded = diag_coded[~diag_coded["dx"].str.startswith(("R", "ZZ"), na=True)]

# Build a 3-character hierarchy stub  e.g. "C10..."
diag_coded["readcode_reduced"] = diag_coded["dx"].str[:3] + "..."

# Observation count per stub
hierarchy_obs = (
    diag_coded.groupby("readcode_reduced")
    .size()
    .rename("obs_count")
    .sort_values(ascending=False)
)

# Patient count per stub
hierarchy_pat = (
    diag_coded.groupby("readcode_reduced")["patient_id"]
    .nunique()
    .rename("patient_count")
    .sort_values(ascending=False)
)

pd.concat([hierarchy_obs, hierarchy_pat], axis=1).head(10)

,obs_count,patient_count
readcode_reduced,,
A10...,3,2
C10...,3,3
44J...,1,1
E12...,1,1
F45...,1,1


## 6. Therapy Drug-Substance Summary

The R pipeline grouped `prodcode` by `drugsubstance.updated` (from `product.csv`) and counted observations and unique patients per substance.

In [10]:
drug_summary = (
    therapy
    .groupby("ingredient", dropna=False)
    .agg(
        obs_count=("patient_id", "count"),
        patient_count=("patient_id", "nunique"),
    )
    .sort_values("patient_count", ascending=False)
)
drug_summary

,obs_count,patient_count
ingredient,,
atorvastatin,1,1
insulin glargine,1,1
metformin,2,1


## 7. Qualified-Patient Filtering

The original CPRD study applied the following eligibility rules before any analysis. These map directly to pandas boolean masks.

| Rule | Meaning |
|---|---|
| `frd ∈ [1918-01-01, 2018-12-31)` | Valid first-registration date |
| `frd ≤ tod` or `tod` is null | Not transferred out before registration |
| `tod ∈ [1918-01-01, 2018-12-31)` or null | Valid transfer-out date |
| `frd < deathdate` or `deathdate` is null | Died after registration |
| `dob < deathdate` or `deathdate` is null | Died after birth |
| `yob + 1800 ≥ 1887` | Born after 1887 (data quality) |
| `gender ≠ 3` | Known gender |

> **Note on `yob`**: CPRD stores year of birth as `year – 1800`, so `yob = 155` corresponds to 1955.

In [11]:
_FMT = "%d/%m/%Y"

qp = patient.copy()
qp["frd"]       = pd.to_datetime(qp["frd"],       format=_FMT, errors="coerce")
qp["tod"]       = pd.to_datetime(qp["tod"],        format=_FMT, errors="coerce")
qp["deathdate"] = pd.to_datetime(qp["deathdate"],  format=_FMT, errors="coerce")
qp["dob"]       = pd.to_datetime((qp["yob"] + 1800).astype(str) + "-01-01", errors="coerce")

frd_ok        = qp["frd"].between("1918-01-01", "2018-12-30")
tod_order_ok  = (qp["frd"] <= qp["tod"]) | qp["tod"].isna()
tod_range_ok  = qp["tod"].between("1918-01-01", "2018-12-30") | qp["tod"].isna()
death_frd_ok  = (qp["frd"]  < qp["deathdate"]) | qp["deathdate"].isna()
death_dob_ok  = (qp["dob"]  < qp["deathdate"]) | qp["deathdate"].isna()
yob_ok        = (qp["yob"] + 1800) >= 1887
gender_ok     = qp["gender"] != 3

qualified = qp[frd_ok & tod_order_ok & tod_range_ok & death_frd_ok & death_dob_ok & yob_ok & gender_ok].drop_duplicates(subset="patid")

print(f"Total patients    : {len(qp):,}")
print(f"Qualified patients: {len(qualified):,}")
qualified[["patid", "yob", "gender", "frd", "dob"]]

Total patients    : 3
Qualified patients: 3


,patid,yob,gender,frd,dob
0,P001,155,2,2000-01-01,1955-01-01
1,P002,162,1,1998-06-01,1962-01-01
2,P003,178,2,2010-03-15,1978-01-01


## 8. Converting to EHRData

Once the canonical tables are built and patients are filtered, the final step is loading everything into an `EHRData` object for downstream analysis.

In [12]:
from ehrdata.io.source import to_ehrdata

# Restrict to qualified patients
qualified_ids = set(qualified["patid"])
patinfo_q  = patinfo[patinfo["patient_id"].isin(qualified_ids)]
diag_q     = diag[diag["patient_id"].isin(qualified_ids)]
therapy_q  = therapy[therapy["patient_id"].isin(qualified_ids)]

edata = to_ehrdata(
    patinfo_q,
    diagnosis=diag_q,
    therapy=therapy_q,
    source="cprd",
)
edata

EHRData object with n_obs × n_vars = 3 × 8
    obs: 'dobyr', 'sex'
    var: 'concept_source', 'concept_code'
    uns: 'source_io_source', 'source_io_tables'
    shape of .X: (3, 8)

In [13]:
print("Patients (obs):")
print(edata.obs)
print()
print("Concepts (var):")
print(edata.var)
print()
print("Presence matrix X:")
print(edata.X)

Patients (obs):
            dobyr     sex
patient_id               
P001          155  Female
P002          162    Male
P003          178  Female

Concepts (var):
                         concept_source      concept_code
concept                                                  
diagnosis:44J3.00             diagnosis           44J3.00
diagnosis:A10..00             diagnosis           A10..00
diagnosis:C10..00             diagnosis           C10..00
diagnosis:E12..00             diagnosis           E12..00
diagnosis:F45..00             diagnosis           F45..00
therapy:atorvastatin            therapy      atorvastatin
therapy:insulin glargine        therapy  insulin glargine
therapy:metformin               therapy         metformin

Presence matrix X:
[[1. 1. 1. 0. 0. 0. 0. 1.]
 [0. 1. 1. 0. 0. 0. 1. 0.]
 [0. 0. 1. 1. 1. 1. 0. 0.]]


## Summary

| Step | Key function |
|---|---|
| Diagnosis (Clinical + Referral + Test) | `cprd.build_diagnosis(..., medical_map=medical_map)` |
| Therapy (prodcode → drug substance) | `cprd.build_therapy(..., product_map=product_map)` |
| Lab tests (Clinical + Additional + Test) | `cprd.build_labtest(clinical, additional, test_data)` |
| Patient info | `cprd.build_patinfo(patient)` |
| Qualified-patient filter | pandas boolean masks on `frd`, `tod`, `yob`, `gender` |
| EHRData bridge | `to_ehrdata(patinfo, diagnosis=..., therapy=...)` |

**Next steps**: See `source_loinc_mapping.ipynb` for LOINC-based lab filtering, or `source_icd_mapping.ipynb` for ICD code mapping workflows.